# LAB 09 — Tokenizer Alignment

Quan sát tokenizer trên CoNLL-2003, thống kê ít nhất 20 câu, rồi kiểm tra alignment giữa word-level labels và subword tokens. Alignment dùng first-subword label; special tokens và continuation subwords nhận -100.

## 1. Imports, model và dataset

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from collections import Counter
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

DATASET_NAME = "lhoestq/conll2003"
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"
MAX_LENGTH = 128
LABEL_NAMES = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
label2id = {label: i for i, label in enumerate(LABEL_NAMES)}
dataset = load_dataset(DATASET_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Dataset splits:", {name: len(split) for name, split in dataset.items()})
print("Tokenizer fast:", tokenizer.is_fast, "| vocab:", tokenizer.vocab_size)
print("NER labels:", LABEL_NAMES)

C:\Users\KIEN\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset splits: {'train': 14041, 'validation': 3250, 'test': 3453}
Tokenizer fast: True | vocab: 30522
NER labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


## 2. Kiểm tra một câu word-level

In [2]:
sample = dataset["train"][0]
words = sample["tokens"]
word_labels = [LABEL_NAMES[tag] for tag in sample["ner_tags"]]
print("Words:", words)
print("Labels:", word_labels)
print("Số words:", len(words), "| số labels:", len(word_labels))

Words: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Labels: ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']
Số words: 9 | số labels: 9


## 3. Experiment 1 — Tokenization trên 20 câu

In [3]:
examples = [dataset["train"][i] for i in range(20)]
rows = []
for sentence_id, example in enumerate(examples):
    words = example["tokens"]
    encoded = tokenizer(words, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
    word_ids = encoded.word_ids()
    counts = Counter(wid for wid in word_ids if wid is not None)
    entity_word_ids = {i for i, tag in enumerate(example["ner_tags"]) if tag != 0}
    split_word_ids = {wid for wid, count in counts.items() if count > 1}
    rows.append({
        "sentence_id": sentence_id,
        "n_words": len(words),
        "n_subword_tokens": sum(counts.values()),
        "n_subword_splits": len(split_word_ids),
        "R_tokens_per_word": sum(counts.values()) / max(1, len(words)),
        "entity_words": len(entity_word_ids),
        "split_entity_words": len(split_word_ids & entity_word_ids),
        "split_non_entity_words": len(split_word_ids - entity_word_ids),
    })
tokenization_table = pd.DataFrame(rows)
display(tokenization_table)

,sentence_id,n_words,n_subword_tokens,n_subword_splits,R_tokens_per_word,entity_words,split_entity_words,split_non_entity_words
0,0,9,9,0,1.000000,3,0,0
1,1,2,2,0,1.000000,2,0,0
2,2,2,6,1,3.000000,1,0,1
3,3,30,31,1,1.033333,4,0,1
4,4,31,37,4,1.193548,6,1,3
5,5,33,39,4,1.181818,5,1,3
6,6,25,25,0,1.000000,2,0,0
7,7,40,47,3,1.175000,3,1,2
8,8,28,41,7,1.464286,8,6,1
9,9,37,41,3,1.108108,2,1,2


### Tóm tắt theo câu ngắn/dài và entity/non-entity

In [4]:
summary = tokenization_table.agg({
    "n_words": ["mean", "min", "max"],
    "n_subword_tokens": ["mean", "min", "max"],
    "n_subword_splits": ["mean", "min", "max"],
    "R_tokens_per_word": ["mean", "min", "max"],
})
display(summary)
print("Tổng entity words bị split:", int(tokenization_table["split_entity_words"].sum()))
print("Tổng non-entity words bị split:", int(tokenization_table["split_non_entity_words"].sum()))

entity_words=int(tokenization_table["entity_words"].sum())
split_entities=int(tokenization_table["split_entity_words"].sum())
non_entity_words=int((tokenization_table["n_words"]-tokenization_table["entity_words"]).sum())
split_non_entities=int(tokenization_table["split_non_entity_words"].sum())
word_group_table=pd.DataFrame([
    {"word_group":"entity","n_words":entity_words,"n_split":split_entities,"split_rate":split_entities/max(1,entity_words)},
    {"word_group":"non-entity","n_words":non_entity_words,"n_split":split_non_entities,"split_rate":split_non_entities/max(1,non_entity_words)},
])
display(word_group_table)

median_words=tokenization_table["n_words"].median()
length_groups=tokenization_table.assign(
    length_group=tokenization_table["n_words"].map(lambda n:"short" if n<=median_words else "long")
)
length_group_table=length_groups.groupby("length_group").agg(
    n_sentences=("sentence_id","count"),
    mean_words=("n_words","mean"),
    mean_subword_tokens=("n_subword_tokens","mean"),
    mean_R=("R_tokens_per_word","mean"),
    total_word_splits=("n_subword_splits","sum"),
).reset_index()
display(length_group_table)

,n_words,n_subword_tokens,n_subword_splits,R_tokens_per_word
mean,23.45,26.85,2.0,1.224316
min,1.00,1.00,0.0,1.000000
max,40.00,47.00,7.0,3.000000


Tổng entity words bị split: 15
Tổng non-entity words bị split: 25


,word_group,n_words,n_split,split_rate
0,entity,68,15,0.220588
1,non-entity,401,25,0.062344


,length_group,n_sentences,mean_words,mean_subword_tokens,mean_R,total_word_splits
0,long,10,33.7,38.0,1.134071,27
1,short,10,13.2,15.7,1.314562,13


**Nhận xét Experiment 1:**  

Trong 20 câu, pooled R=1,145; trung bình sentence-level R=1,224. Có 15/68 entity words (22,1%) và 25/401 non-entity words (6,2%) bị split. Trong sample này câu ngắn có mean R=1,315, câu dài=1,134; đây là sample nhỏ nên chưa khái quát thành quy luật.

## 4. Xem WordPiece tokens và word_ids

In [5]:
# Chọn một trong 20 câu có subword split để xem chi tiết.
chosen = None
for example in examples:
    encoded = tokenizer(example["tokens"], is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
    ids = encoded.word_ids()
    counts = Counter(wid for wid in ids if wid is not None)
    if any(count > 1 for count in counts.values()):
        chosen = example
        break
if chosen is None:
    chosen = examples[0]

encoded = tokenizer(chosen["tokens"], is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
word_ids = encoded.word_ids()
subwords = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print("Words:", chosen["tokens"])
print("Subwords:", subwords)
print("word_ids:", word_ids)
print("attention_mask:", encoded["attention_mask"])

Words: ['BRUSSELS', '1996-08-22']
Subwords: ['[CLS]', 'brussels', '1996', '-', '08', '-', '22', '[SEP]']
word_ids: [None, 0, 1, 1, 1, 1, 1, None]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1]


## 5. Alignment — first subword gets the word label

In [6]:
word_labels = chosen["ner_tags"]
aligned_labels = []
previous_word_id = None
for word_id in word_ids:
    if word_id is None:
        aligned_labels.append(-100)
    elif word_id != previous_word_id:
        aligned_labels.append(word_labels[word_id])
    else:
        aligned_labels.append(-100)
    previous_word_id = word_id

alignment_rows = []
for token_id, subword, word_id, label_id in zip(encoded["input_ids"], subwords, word_ids, aligned_labels):
    alignment_rows.append({
        "token_id": token_id,
        "subword": subword,
        "word_id": word_id,
        "original_word": "[special]" if word_id is None else chosen["tokens"][word_id],
        "word_label": "IGNORE" if word_id is None else LABEL_NAMES[word_labels[word_id]],
        "aligned_label": "IGNORE" if label_id == -100 else LABEL_NAMES[label_id],
    })
alignment_table = pd.DataFrame(alignment_rows)
display(alignment_table)
print("IGNORE positions:", sum(label == -100 for label in aligned_labels))

,token_id,subword,word_id,original_word,word_label,aligned_label
0,101,[CLS],NaN,[special],IGNORE,IGNORE
1,9371,brussels,0.0,BRUSSELS,B-LOC,B-LOC
2,2727,1996,1.0,1996-08-22,O,O
3,1011,-,1.0,1996-08-22,O,IGNORE
4,5511,08,1.0,1996-08-22,O,IGNORE
5,1011,-,1.0,1996-08-22,O,IGNORE
6,2570,22,1.0,1996-08-22,O,IGNORE
7,102,[SEP],NaN,[special],IGNORE,IGNORE


IGNORE positions: 6


## 6. Kiểm tra tính nhất quán

In [7]:
assert len(encoded["input_ids"]) == len(aligned_labels) == len(word_ids)
print("input_ids, word_ids và aligned labels có cùng số vị trí:", len(aligned_labels))
print("Strategy:", "nhãn ở subword đầu; special/continuation subwords = -100")

input_ids, word_ids và aligned labels có cùng số vị trí: 8
Strategy: nhãn ở subword đầu; special/continuation subwords = -100


**Ghi chú sau khi quan sát:**  

Với `1996-08-22`, tokenizer tạo 5 pieces; nhãn được giữ ở piece đầu và bốn continuation pieces cùng hai special tokens bị ignore trong loss.